# Filter bike-eligible infrastructure out of scraped folder

In [1]:
import os
from convenient_pickle import *

import networkx as nx
import gc
import shapely.geometry as geometry
from shapely.ops import linemerge, unary_union, polygonize
from shapely.geometry import mapping
import geojson
import haversine

## 1. Load the source dataset

In [2]:
def load_dataset(prefix, base_dir="pickle_folder"):
    """
    Load a previously saved scrape run (graph, nodes, ways, used bboxes)
    using the same file layout as save_the_pickles().
    """
    folder = os.path.join(base_dir, prefix)
    G = load_pickle(os.path.join(folder, f"{prefix}_graph.pkl"))
    total_result_nodes = load_pickle(os.path.join(folder, f"{prefix}_total_result_nodes.pkl"))
    total_result_ways = load_pickle(os.path.join(folder, f"{prefix}_total_result_ways.pkl"))
    used_bboxes = load_pickle(os.path.join(folder, f"{prefix}_used_bboxes.pkl"))
    return G, total_result_nodes, total_result_ways, used_bboxes


## 2. Decide which ways are bike-eligible

This mirrors the tag logic from `get_bike_query()` exactly, just applied
locally to `way.tags` instead of as an Overpass filter:
- `highway=cycleway`
- `highway=path` with `bicycle in (designated, yes)`
- `footway=path` with `bicycle=yes`
- `footway=crossing`
- `highway=crossing`

In [3]:
def is_bike_eligible(way):
    tags = way.tags
    highway = tags.get("highway")
    footway = tags.get("footway")
    bicycle = tags.get("bicycle")

    if highway == "cycleway":
        return True
    if footway == "path" and bicycle == "yes":
        return True
    if highway == "crossing" and bicycle == "yes":
        return True
    if bicycle in ('designated', 'yes'):
        return True
    return False


def filter_bike_ways(total_result_ways, min_way_len=2):
    return [way for way in total_result_ways if is_bike_eligible(way) and len(way.nodes) >= min_way_len]


## 3. Rebuild nodes to match the filtered ways

In [4]:
def filter_nodes_to_ways(nodes, ways):
    used_node_ids = set()
    for way in ways:
        for node in way.nodes:
            used_node_ids.add(node.id)

    return [node for node in nodes if node.id in used_node_ids]


## 4. Rebuild the graph from the filtered ways

Same unweighted node/edge graph as the original scraper's `make_graph()`,
just run over the bike-only way list.

In [5]:
def build_graph_from_ways(ways):
    G = nx.Graph()
    for way in ways:
        nodes = way.nodes
        for node_dex in range(len(nodes)):
            node_id = nodes[node_dex].id
            if not G.has_node(node_id):
                G.add_node(node_id)
            if node_dex > 0:
                G.add_edge(node_id, nodes[node_dex - 1].id)
    return G


## 5. Remove insignificant ways (ie, crossings that don't connect to bike infrastructure, user errors, etc) from system

In [6]:
def remove_insignificant(G_bike, total_result_ways_bike, threshold=1):
    bike_ccs = sorted([i for i in nx.connected_components(G_bike)], key = lambda x: -len(x))
    bike_cc_dict = dict()
    
    for cc in bike_ccs:
        cclen = len(cc)
        if cclen <= threshold: 
            break
        for node in cc: 
            bike_cc_dict[node]= cclen
    new_total_result_ways_bike = []
    new_total_result_nodes_bike = []
    for way in total_result_ways_bike: 
        found = False
        for node in way.nodes: 
            if node.id in bike_cc_dict.keys() and not found: 
                new_total_result_ways_bike.append(way)
                new_total_result_nodes_bike.append(node)
                found = True
            elif node.id in bike_cc_dict.keys(): 
                new_total_result_nodes_bike.append(node)
                
    return new_total_result_ways_bike, new_total_result_nodes_bike

## 6. Only include LCC bikeways

In [7]:
def remove_non_lcc(G, bike_ways):
    lcc = max(nx.connected_components(G), key=len)
    lcc = G.subgraph(lcc).copy()
    out_ways = []
    out_nodes = []
    for way in bike_ways: 
        for node in way.nodes: 
            if node.id in lcc.nodes: 
                out_ways.append(way)
                out_nodes+= way.nodes
                break
    return out_ways, out_nodes

## 6. Save the filtered dataset


In [8]:
def save_dataset(prefix, G, total_result_nodes, total_result_ways, used_bboxes, base_dir="pickle_folder", warn = True):
    folder = os.path.join(base_dir, prefix)
    os.makedirs(folder, exist_ok=True)

    dump_pickle(folder, f"{prefix}_graph.pkl", G, warn=warn)
    dump_pickle(folder, f"{prefix}_total_result_nodes.pkl", total_result_nodes, warn=warn)
    dump_pickle(folder, f"{prefix}_total_result_ways.pkl", total_result_ways, warn=warn)
    dump_pickle(folder, f"{prefix}_used_bboxes.pkl", used_bboxes, warn=warn)

    print(f"Saved filtered dataset to {folder}")


## 7. Run it

In [9]:
SOURCE_PREFIX = "06_26_2026"
TARGET_PREFIX = "06_26_2026_just_bikes"

G, total_result_nodes, total_result_ways, used_bboxes = load_dataset(SOURCE_PREFIX)

bike_ways = filter_bike_ways(total_result_ways)
bike_nodes = filter_nodes_to_ways(total_result_nodes, bike_ways)
bike_graph = build_graph_from_ways(bike_ways)
new_bike_ways, new_bike_nodes = remove_insignificant(bike_graph, bike_ways)

del total_result_nodes
del bike_ways
del bike_nodes
gc.collect()

new_bike_ways, new_bike_nodes = remove_non_lcc(G, new_bike_ways) 
final_graph = build_graph_from_ways(new_bike_ways)



print(f"Source ways: {len(total_result_ways)} -> bike-eligible ways: {len(new_bike_ways)}")
#print(f"Source nodes: {len(total_result_nodes)} -> bike-eligible nodes: {len(bike_nodes)}")

save_dataset(TARGET_PREFIX, final_graph, new_bike_nodes, new_bike_ways, used_bboxes, warn=False)


Source ways: 481654 -> bike-eligible ways: 16295
Saved filtered dataset to pickle_folder/06_26_2026_just_bikes


# 8. Save geojson file

## Begin by deleting unnecessary variables

In [12]:
del G
del used_bboxes
del final_graph
del bike_graph

gc.collect()

432

In [13]:
def convert_to_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    borders = unary_union(merged) # linestrings to a MultiLineString
    polygons = list(polygonize(borders))
    return merged, borders, polygons


In [15]:
bike_merged, bike_borders, bike_polygons = convert_to_borders(new_bike_ways)
out_bike_geojson= mapping(bike_merged)

In [16]:
with open(f'geojsons/bike_only_geojson.geojson', 'w') as file:
    geojson.dump(out_bike_geojson, file)